# 02 – Feature Engineering
**Project:** War Economic Impact Predictor  

## Objectives
1. Run the full preprocessing pipeline to produce a clean parquet file  
2. Apply advanced feature engineering (log transforms, interactions, composite scores)  
3. Assess feature quality via mutual information and variance analysis  
4. Save final feature matrix for modelling

---

In [ ]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

ROOT = Path().resolve().parent
sys.path.insert(0, str(ROOT))

from src.data.preprocess import DataPreprocessor
from src.features.build_features import FeatureEngineer
from src.utils import load_config

%matplotlib inline
plt.rcParams['figure.dpi'] = 110
cfg = load_config(ROOT / 'config' / 'config.yaml')
print('Config loaded.')

## 1. Run Preprocessing Pipeline

In [ ]:
preprocessor = DataPreprocessor(cfg)

raw_path = ROOT / cfg['paths']['raw_data']
processed_path = ROOT / cfg['paths']['processed_data']

df_raw = preprocessor.load(raw_path)
df_clean = preprocessor.clean(df_raw)
df_proc = preprocessor.transform(df_clean)
preprocessor.save(df_proc, processed_path)

print(f'Processed shape: {df_proc.shape}')
df_proc.head(3)

In [ ]:
# Severity label distribution check
label_map = {0: 'Mild', 1: 'Moderate', 2: 'Severe', 3: 'Catastrophic'}
sev_dist = df_proc['Severity_Label'].map(label_map).value_counts()
print(sev_dist)

## 2. Advanced Feature Engineering

In [ ]:
engineer = FeatureEngineer(cfg)
df_feat = engineer.build(df_proc)
print(f'Feature matrix shape: {df_feat.shape}')
print('New columns added:')
new_cols = [c for c in df_feat.columns if c not in df_proc.columns]
print(new_cols)

In [ ]:
df_feat[new_cols].describe().T

## 3. Mutual Information Feature Ranking

In [ ]:
# Mutual information vs regression target
mi_scores = engineer.mutual_info_ranking(df_feat, target='GDP_Change_%', top_n=25)

fig, ax = plt.subplots(figsize=(9, 7))
mi_scores.sort_values().plot(kind='barh', color='steelblue', ax=ax)
ax.set_xlabel('Mutual Information Score')
ax.set_title('Top 25 Features – Mutual Information with GDP Change (%)')
plt.tight_layout()
plt.savefig(ROOT / 'reports' / 'figures' / 'mutual_info_regression.png', dpi=150)
plt.show()

In [ ]:
from sklearn.feature_selection import mutual_info_classif

feat_cols = engineer.get_feature_columns(df_feat, 'Severity_Label')
X = df_feat[feat_cols].fillna(0)
y = df_feat['Severity_Label']

mi_cls = pd.Series(
    mutual_info_classif(X, y, random_state=42),
    index=feat_cols
).sort_values(ascending=False).head(25)

fig, ax = plt.subplots(figsize=(9, 7))
mi_cls.sort_values().plot(kind='barh', color='coral', ax=ax)
ax.set_xlabel('Mutual Information Score')
ax.set_title('Top 25 Features – Mutual Information with Severity Label')
plt.tight_layout()
plt.savefig(ROOT / 'reports' / 'figures' / 'mutual_info_classification.png', dpi=150)
plt.show()

## 4. Variance Analysis (Low-Variance Filter)

In [ ]:
from sklearn.feature_selection import VarianceThreshold

feat_cols = engineer.get_feature_columns(df_feat, 'GDP_Change_%')
X = df_feat[feat_cols].fillna(0)

sel = VarianceThreshold(threshold=0.01)
sel.fit(X)
low_var = [c for c, keep in zip(feat_cols, sel.get_support()) if not keep]
print(f'Low-variance features (threshold=0.01): {low_var}')

variances = pd.Series(X.var(), name='Variance').sort_values(ascending=False)
fig, ax = plt.subplots(figsize=(9, 6))
variances.head(25).plot(kind='bar', color='teal', ax=ax)
ax.set_title('Top 25 Features by Variance')
ax.set_ylabel('Variance')
plt.xticks(rotation=45, ha='right', fontsize=8)
plt.tight_layout()
plt.show()

In [ ]:
# ── Save feature matrix ───────────────────────────────────────────────────────
out_path = ROOT / 'data' / 'processed' / 'war_economic_features.parquet'
df_feat.to_parquet(out_path, index=False)
print(f'Feature matrix saved → {out_path}  {df_feat.shape}')

## 5. Feature Engineering Summary

| Feature Group | Examples | Count |
|---|---|---|
| Raw numeric | Pre_War_Unemployment_%, Inflation_Rate_% | 17 |
| Raw categorical (encoded) | Conflict_Type, Region | 5 |
| Log transforms | log_Cost_of_War_USD | 3 |
| Ratio interactions | During/Pre unemployment ratio | 6 |
| Composite scores | Economic_Stress_Index, Black_Market_Pressure | 2 |
| Duration | Conflict_Duration_Years | 1 |

> **Next step:** `03_modeling.ipynb`